In [37]:
"""
Social Text Analytics — Climate Change Reddit Comments
Covers: Keyword frequency, TF-IDF, VADER sentiment, Controversiality analysis
"""

import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings("ignore")


In [38]:

# ── Config ─────────────────────────────────────────────────────────────────────
DATA_PATH = "C:\\Users\\Hungthq\\MITB\\ISSS606_ADS_in_Social_Networks\\Group_Project\\GIT\\DATASETS\\"
OUT_DIR   = "C:\\Users\\Hungthq\\MITB\\ISSS606_ADS_in_Social_Networks\\Group_Project\\GIT\\PROCESSED\\"
os.makedirs(OUT_DIR, exist_ok=True)


In [66]:
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english')).union({
    "like","just","people","know","think","really","good","time",
    "see","make","want","need","lot","things","thing","time","years",
    "year","2024","2025","2026", "would","could","also","even","much","many","get","got","one", "any", 
    "most", "other", "still", "way", "may", "might", "must", "may", "might", "must", "may", "might", "must",
})

In [39]:

STOPWORDS = {
    "the","a","an","is","it","in","on","at","to","for","of","and","or","but",
    "be","are","was","were","been","not","that","this","with","from","they",
    "have","has","had","he","she","we","you","i","my","your","our","their",
    "its","by","do","did","does","so","if","as","up","out","no","can","will",
    "just","like","get","go","got","also","more","than","then","what","how",
    "all","one","even","would","could","should","about","some","there","when",
    "which","who","his","her","him","them","me","us","s","t","re","ve","ll",
    "amp","gt","lt","www","http","https","reddit","subreddit","comment","post",
    "really","very","much","well","good","bad","think","know","people","don",
    "doesn","isn","won","said","say","says","still","way","many","make","made",
    "need","want","lot","things","thing","time","years","year","2024","2025","2026",
}

#FOCUS_SUBREDDITS = [
#    "climateskeptics","climatechange","climate","ClimateShitposting",
#    "ClimateOffensive","ClimateActionPlan","GlobalClimateChange",
#    "ClimateMemes","CitizensClimateLobby","Climate_Nuremberg","ClimateCO",
#    "conspiracy","science","politics","worldnews","environment",
#]



In [40]:
#Define a focused list of subreddits to analyze, based on relevance and activity
FOCUS_SUBREDDITS = [
    'conspiracy', 'changemyview', 'climatechange', 'energy', 'climate', 
    'Futurology', 'politics', 'europe', 'worldnews', 'science', 
    'ClimateShitposting', 'canada', 'unitedkingdom', 'news', 'climateskeptics']

In [67]:

def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    return text.lower().strip()

def tokenize(text):
    return [w for w in clean_text(text).split() if w not in STOPWORDS and len(w) > 2]

def save(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


In [68]:

# ── Load ───────────────────────────────────────────────────────────────────────
print("Loading data …")
df = pd.read_csv(DATA_PATH + "cleaned_climate_comments_2025_05_to_2026_04.xls", usecols=[
    "comment_id","score","self_text","subreddit","controversiality",
    "comment_month","comment_word_count","post_title"
])
df["self_text"] = df["self_text"].fillna("")
df["controversial"] = df["controversiality"].astype(int)
df["comment_month"] = pd.to_datetime(df["comment_month"], errors="coerce")
print(f"  {len(df):,} rows, {df['subreddit'].nunique()} subreddits")


Loading data …
  569,506 rows, 24 subreddits


In [70]:
# ── Lemmatization Pipeline ───────────────────────────────────────────────────

import spacy

# Load the lightweight English language model
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def lemmatize_pipeline(text):
    if not isinstance(text, str):
        return []
    
    # Process the text
    doc = nlp(text.lower())
    
    clean_tokens = []
    for token in doc:
        # 1. Strip out punctuation, spaces, and standard structural stopwords
        if token.is_punct or token.is_space or token.is_stop:
            continue
            
        # 2. Get the base dictionary form of the word (.lemma_)
        lemma = token.lemma_.strip()
        
        # 3. Guard against residual artifacts or short words
        if len(lemma) > 1:
            clean_tokens.append(lemma)
            
    return clean_tokens


In [73]:

# Sample for tokenisation-heavy steps
SAMPLE = 80_000
sample_df = df.sample(SAMPLE, random_state=42).copy()
print(f"  Working sample for sentiment/keywords: {SAMPLE:,}")

# Tokenize sample
print("  Tokenizing …")
sample_df["tokens"] = sample_df["self_text"].apply(tokenize)

# Apply this to your dataframe to generate your new, consolidated tokens
sample_df["lemma_tokens"] = sample_df["tokens"].apply(lemmatize_pipeline)

# Also tokenize controversial/non-controversial subsets from full df
# (do only 30K per group to stay fast)
cont1 = df[df["controversial"] == 1].sample(min(15000, df["controversial"].sum()), random_state=1)
cont0 = df[df["controversial"] == 0].sample(15000, random_state=2)
cont1["tokens"] = cont1["self_text"].apply(tokenize)
cont0["tokens"] = cont0["self_text"].apply(tokenize)


  Working sample for sentiment/keywords: 80,000
  Tokenizing …


In [96]:

# ── 1. Global keyword frequency ────────────────────────────────────────────────
print("\n[1] Keyword frequency …")
all_tokens = [t for ts in sample_df["tokens"] for t in ts]
freq       = Counter(all_tokens)
top_n      = freq.most_common(30)

words, counts = zip(*top_n)
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(words[::-1], counts[::-1], color=sns.color_palette("Blues_d", 30))
ax.set_xlabel("Frequency", fontsize=11)
ax.set_title("Top 30 Keywords — All Subreddits (80K sample)", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
plt.tight_layout()
plt.show()
save(fig, "01_keyword_frequency_global.png")

# Per-subreddit keyword heatmap
print("  Per-subreddit keyword heatmap …")
sub_df = sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]
sub_freqs = {}
for sub, grp in sub_df.groupby("subreddit"):
    toks = [t for ts in grp["tokens"] for t in ts]
    #toks = [t for ts in grp["lemma_tokens"] for t in ts]
    
    sub_freqs[sub] = Counter(toks)

sub_all = Counter()
for c in sub_freqs.values(): sub_all.update(c)
top_sub_words = [w for w, _ in sub_all.most_common(20)]

heat_data = pd.DataFrame(
    {sub: [sub_freqs[sub].get(w, 0) for w in top_sub_words] for sub in sub_freqs},
    index=top_sub_words,
).T
heat_norm = heat_data.div(heat_data.sum(axis=1).replace(0,1), axis=0) * 1000

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(heat_norm, cmap="YlOrRd", linewidths=0.3, ax=ax,
            cbar_kws={"label": "Relative freq (‰)"})
ax.set_title("Keyword Relative Frequency per Subreddit (top 20 terms)", fontsize=13, fontweight="bold")
ax.set_xlabel("Keyword"); ax.set_ylabel("Subreddit")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
save(fig, "02_keyword_heatmap_subreddits.png")



[1] Keyword frequency …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\01_keyword_frequency_global.png
  Per-subreddit keyword heatmap …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\02_keyword_heatmap_subreddits.png


In [93]:

# ── 2. TF-IDF per subreddit ────────────────────────────────────────────────────
print("\n[2] TF-IDF per subreddit …")
sub_docs = (
    sub_df.groupby("subreddit")["self_text"]
    .apply(lambda x: " ".join(x.dropna().apply(clean_text)))
    .reset_index()
)
sub_docs.columns = ["subreddit", "text_clean"]

vectorizer = TfidfVectorizer(
    max_features=5000, stop_words=list(STOPWORDS),
    ngram_range=(1, 2), min_df=2,
)
tfidf_matrix = vectorizer.fit_transform(sub_docs["text_clean"])
feature_names = vectorizer.get_feature_names_out()

tfidf_records = []
for i, sub in enumerate(sub_docs["subreddit"]):
    row     = tfidf_matrix[i].toarray().flatten()
    top_idx = row.argsort()[::-1][:10]
    for rank, idx in enumerate(top_idx):
        tfidf_records.append({"subreddit": sub, "rank": rank+1,
                               "term": feature_names[idx], "tfidf": round(float(row[idx]),5)})

tfidf_df = pd.DataFrame(tfidf_records)
tfidf_df.to_csv(os.path.join(OUT_DIR, "03_tfidf_top10_per_subreddit.csv"), index=False)
print("  Saved → figures/03_tfidf_top10_per_subreddit.csv")

plot_subs = [s for s in FOCUS_SUBREDDITS if s in sub_docs["subreddit"].values][:12]
n_cols = 4
n_rows = int(np.ceil(len(plot_subs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.2))
axes = axes.flatten()
palette = sns.color_palette("tab20", 10)

for ax, sub in zip(axes, plot_subs):
    sub_data = tfidf_df[tfidf_df["subreddit"] == sub].head(8)
    if sub_data.empty: ax.set_visible(False); continue
    ax.barh(sub_data["term"][::-1].values, sub_data["tfidf"][::-1].values, color=palette)
    ax.set_title(f"r/{sub}", fontsize=9, fontweight="bold")
    ax.tick_params(axis="y", labelsize=7); ax.tick_params(axis="x", labelsize=7)
for ax in axes[len(plot_subs):]: ax.set_visible(False)
fig.suptitle("Top TF-IDF Terms per Subreddit (unigrams + bigrams)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
save(fig, "03_tfidf_facets.png")



[2] TF-IDF per subreddit …
  Saved → figures/03_tfidf_top10_per_subreddit.csv
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\03_tfidf_facets.png


In [94]:

# ── 3. VADER Sentiment ─────────────────────────────────────────────────────────
print("\n[3] VADER sentiment …")
analyzer = SentimentIntensityAnalyzer()
print(f"  Scoring {SAMPLE:,} comments …")

scores = sample_df["self_text"].apply(
    lambda t: analyzer.polarity_scores(t if isinstance(t, str) else "")
)
sample_df["vader_neg"]      = scores.apply(lambda s: s["neg"])
sample_df["vader_pos"]      = scores.apply(lambda s: s["pos"])
sample_df["vader_compound"] = scores.apply(lambda s: s["compound"])
sample_df["sentiment_label"] = pd.cut(
    sample_df["vader_compound"],
    bins=[-1.01, -0.05, 0.05, 1.01],
    labels=["Negative", "Neutral", "Positive"],
)

# 3a. Global distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(sample_df["vader_compound"], bins=60, color="#4C72B0", edgecolor="white", linewidth=0.3)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.2, label="Neutral boundary")
axes[0].set_xlabel("VADER Compound Score", fontsize=11); axes[0].set_ylabel("Count", fontsize=11)
axes[0].set_title("Distribution of VADER Compound Scores", fontsize=12, fontweight="bold")
axes[0].legend()

label_counts = sample_df["sentiment_label"].value_counts()
axes[1].pie(label_counts, labels=label_counts.index, autopct="%1.1f%%",
            colors=["#d62728","#aec7e8","#2ca02c"], startangle=140, textprops={"fontsize":11})
axes[1].set_title("Sentiment Label Distribution", fontsize=12, fontweight="bold")
plt.tight_layout()
save(fig, "04_vader_global_distribution.png")

# 3b. Mean sentiment by subreddit
print("  Sentiment by subreddit …")
sub_sent = (
    sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby("subreddit")["vader_compound"]
    .agg(["mean","median","std","count"])
    .reset_index()
    .rename(columns={"mean":"mean_compound","median":"median_compound",
                     "std":"std_compound","count":"n_comments"})
    .sort_values("mean_compound")
)
sub_sent.to_csv(os.path.join(OUT_DIR, "04_sentiment_by_subreddit.csv"), index=False)

fig, ax = plt.subplots(figsize=(11, 7))
colors = ["#d62728" if v < 0 else "#2ca02c" for v in sub_sent["mean_compound"]]
ax.barh(sub_sent["subreddit"], sub_sent["mean_compound"], color=colors)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Mean VADER Compound Score", fontsize=11)
ax.set_title("Mean Sentiment by Subreddit", fontsize=13, fontweight="bold")
ax.set_xlim(-0.25, 0.25)
plt.tight_layout()
save(fig, "04_sentiment_by_subreddit.png")

# 3c. Sentiment over time
print("  Sentiment over time …")
monthly = (
    sample_df.dropna(subset=["comment_month"])
    .groupby("comment_month")["vader_compound"]
    .agg(["mean","median"]).reset_index()
)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["comment_month"], monthly["mean"],   marker="o", label="Mean",   color="#4C72B0")
ax.plot(monthly["comment_month"], monthly["median"], marker="s", linestyle="--", label="Median", color="#DD8452")
ax.axhline(0, color="grey", linewidth=0.7, linestyle=":")
ax.set_xlabel("Month", fontsize=11); ax.set_ylabel("VADER Compound Score", fontsize=11)
ax.set_title("Sentiment Trend Over Time", fontsize=13, fontweight="bold"); ax.legend()
plt.xticks(rotation=30); plt.tight_layout()
save(fig, "04_sentiment_trend_monthly.png")

# 3d. Stacked sentiment composition per subreddit
sub_label = (
    sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby(["subreddit","sentiment_label"]).size().unstack(fill_value=0)
)
sub_label_pct = sub_label.div(sub_label.sum(axis=1), axis=0) * 100
sub_label_pct = sub_label_pct.sort_values("Negative", ascending=False)
fig, ax = plt.subplots(figsize=(12, 7))
sub_label_pct.plot(kind="barh", stacked=True, ax=ax, width=0.7,
                   color={"Negative":"#d62728","Neutral":"#aec7e8","Positive":"#2ca02c"})
ax.set_xlabel("% of comments", fontsize=11)
ax.set_title("Sentiment Composition per Subreddit", fontsize=13, fontweight="bold")
ax.legend(title="Sentiment", loc="lower right"); plt.tight_layout()
save(fig, "04_sentiment_stacked_subreddits.png")



[3] VADER sentiment …
  Scoring 80,000 comments …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\04_vader_global_distribution.png
  Sentiment by subreddit …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\04_sentiment_by_subreddit.png
  Sentiment over time …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\04_sentiment_trend_monthly.png
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\04_sentiment_stacked_subreddits.png


In [85]:
#Gemini
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

# ── 1. Global keyword frequency ────────────────────────────────────────────────
print("\n[1] Keyword frequency …")

# Flattening list of lists
all_tokens = [t for ts in sample_df["tokens"] for t in ts]

if not all_tokens:
    print("Warning: 'tokens' column is empty. Skipping global frequency plot.")
else:
    freq = Counter(all_tokens)
    top_n = freq.most_common(30)

    words, counts = zip(*top_n)
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Using 30 distinct values for color palette matches your top_n count
    ax.barh(words[::-1], counts[::-1], color=sns.color_palette("Blues_d", len(top_n)))
    ax.set_xlabel("Frequency", fontsize=11)
    ax.set_title("Top 30 Keywords — All Subreddits (80K sample)", fontsize=13, fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.tight_layout()
    plt.show()
    save(fig, "01_G_keyword_frequency_global.png")

# Per-subreddit keyword heatmap
print("  Per-subreddit keyword heatmap …")
sub_df = sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]

sub_freqs = {}
sub_total_counts = {}  # Track true total words per sub for accurate normalization

for sub, grp in sub_df.groupby("subreddit"):
    toks = [t for ts in grp["tokens"] for t in ts]
    sub_freqs[sub] = Counter(toks)
    sub_total_counts[sub] = len(toks)  # True denominator

# Aggregating across the focused subreddits
sub_all = Counter()
for c in sub_freqs.values(): 
    sub_all.update(c)
top_sub_words = [w for w, _ in sub_all.most_common(20)]

# Build DataFrame
heat_data = pd.DataFrame(
    {sub: [sub_freqs[sub].get(w, 0) for w in top_sub_words] for sub in sub_freqs},
    index=top_sub_words,
).T

# FIX: Normalize by total vocabulary size of the subreddit instead of the top 20 sum
sub_totals_series = pd.Series(sub_total_counts).reindex(heat_data.index).replace(0, 1)
heat_norm = heat_data.div(sub_totals_series, axis=0) * 1000

# Plotting
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(heat_norm, cmap="YlOrRd", linewidths=0.3, ax=ax,
            cbar_kws={"label": "Relative freq per 1,000 tokens (‰)"},
            annot=True, fmt=".1f") # Added annotations for clarity in data analysis
ax.set_title("Keyword Relative Frequency per Subreddit (top 20 terms)", fontsize=13, fontweight="bold")
ax.set_xlabel("Keyword")
ax.set_ylabel("Subreddit")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()
save(fig, "02_G_keyword_heatmap_subreddits.png")



[1] Keyword frequency …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\01_G_keyword_frequency_global.png
  Per-subreddit keyword heatmap …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\02_G_keyword_heatmap_subreddits.png


In [87]:
#Context discovered by Gemini:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from nltk.util import ngrams

# ── 2. Context Discovery (N-Grams Analysis) ───────────────────────────────────
print("\n[2] Extracting contextual phrases (Bigrams/Trigrams) …")

# Define a function to generate valid n-grams from a list of strings
def get_valid_ngrams(token_list, n=2):
    # Standard NLTK tuple generator: [('machine', 'learning'), ('deep', 'learning')]
    generated_ngrams = ngrams(token_list, n)
    
    # Context filter: Filter out phrases that are just repeated single characters or artifacts
    valid_phrases = []
    for gram in generated_ngrams:
        # Reconstruct into a clean text string: "machine learning"
        phrase = " ".join(gram)
        # Skip if the text contains empty strings or weird spacing bugs
        if any(len(word.strip()) <= 1 for word in gram if word.isalnum()): 
            continue
        valid_phrases.append(phrase)
    return valid_phrases

# 1. Choose your N (2 = Bigrams, 3 = Trigrams)
N_GRAM_SIZE = 2  # Set to 3 for structural context like "artificial intelligence model"

# 2. Extract context across the entire token space
all_phrases = []
for tokens in sample_df["tokens"]:
    all_phrases.extend(get_valid_ngrams(tokens, n=N_GRAM_SIZE))

# 3. Aggregate and select top context drivers
phrase_freq = Counter(all_phrases)
top_phrases = phrase_freq.most_common(25)

# 4. Handle plotting boundaries gracefully
if not top_phrases:
    print("No valid phrases found. Check your token lengths.")
else:
    phrases, counts = zip(*top_phrases)
    
    # Plot horizontal visualization for easy phrase readability
    fig, ax = plt.subplots(figsize=(12, 7))
    #ax.barh(phrases[::-1], counts[::-1], color=sns.color_palette("Viridis_r", len(top_phrases)))
    # Replace the ax.barh line with this:
    ax.barh(phrases[::-1], counts[::-1], color=plt.cm.viridis_r(np.linspace(0, 1, len(top_phrases))))
    
    ax.set_xlabel("Phrase Count", fontsize=11)
    ax.set_ylabel("Discovered Context", fontsize=11)
    ax.set_title(f"Top {len(top_phrases)} Multi-Word Context Phrases (N={N_GRAM_SIZE})", 
                 fontsize=13, fontweight="bold")
    
    plt.tight_layout()
    plt.show()
    save(fig, f"03_G_context_discovery_n_{N_GRAM_SIZE}.png")


[2] Extracting contextual phrases (Bigrams/Trigrams) …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\03_G_context_discovery_n_2.png


In [88]:


import itertools
import networkx as nx

# ── 3. Co-occurrence & Network Analysis ─────────────────────────────────────
print("\n[3] Building term co-occurrence network …")

# 1. Identify top global keywords to keep the network clean and readable
# (Using too many nodes creates an unreadable "hairball" graph)
MAX_NODES = 25
all_tokens_flat = [t for ts in sample_df["tokens"] for t in ts]
global_counts = Counter(all_tokens_flat)
top_words = set([w for w, _ in global_counts.most_common(MAX_NODES)])

# 2. Calculate co-occurrences within each comment's token list
co_counts = Counter()

for tokens in sample_df["tokens"]:
    # Filter tokens to only include our top target words
    valid_tokens = sorted(list(set([t for t in tokens if t in top_words])))
    
    # Generate unique pairs within the same comment
    if len(valid_tokens) >= 2:
        for word1, word2 in itertools.combinations(valid_tokens, 2):
            co_counts[(word1, word2)] += 1

if not co_counts:
    print("Warning: No co-occurring word pairs found among the top terms.")
else:
    # 3. Construct the NetworkX Graph
    G = nx.Graph()

    # Add edges with weights (frequencies)
    for (w1, w2), weight in co_counts.items():
        # You can adjust this threshold if the graph is too crowded
        if weight > 5:  
            G.add_edge(w1, w2, weight=weight)

    # 4. Compute layout and visual attributes
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Spring layout positions connected nodes closer together
    pos = nx.spring_layout(G, k=0.4, iterations=50, seed=42)

    # Scale node sizes based on global frequency
    node_sizes = [global_counts[node] * 0.15 for node in G.nodes()]

    # Scale edge weights for visual thickness distinction
    edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
    max_weight = max(edge_weights) if edge_weights else 1
    # Normalize weights to a reasonable line width range (e.g., 1 to 6)
    edge_widths = [1 + (w / max_weight) * 5 for w in edge_weights]

    # 5. Drawing elements
    # Draw edges with varying transparency based on weight strength
    nx.draw_networkx_edges(
        G, pos, ax=ax, 
        width=edge_widths, 
        edge_color="skyblue", 
        alpha=0.6
    )

    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_size=node_sizes,
        node_color=sns.color_palette("viridis", len(G.nodes())),
        alpha=0.85
    )

    # Draw labels with a clean background box for legibility
    nx.draw_networkx_labels(
        G, pos, ax=ax,
        font_size=10,
        font_weight="bold",
        font_family="sans-serif",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=2)
    )

    ax.set_title("Keyword Co-occurrence Network (Semantic Closeness)", 
                 fontsize=14, fontweight="bold")
    ax.axis("off")  # Clear graph axes
    
    plt.tight_layout()
    plt.show()
    save(fig, "04_G_keyword_cooccurrence_network.png")


[3] Building term co-occurrence network …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\04_G_keyword_cooccurrence_network.png



🛠️ Performance & Tuning Adjustments
Managing Noise (weight > 5): Depending on how closely associated your 80K sample data is, you may need to tweak the minimum edge weight filter. If the graph looks like a massive web where everything connects to everything, increase that threshold (e.g., weight > 50) to isolate only the absolute strongest thematic pairings.

Layout Isolation (seed=42): nx.spring_layout relies on a random initialization simulation. Setting a permanent random seed guarantees your nodes won't randomly jump to new spots every time you run the script, which keeps your dashboard consistent.

How are the keyword connections mapping out? Do you see distinct standalone clusters forming, or does a single giant master node dominate the entire chart?

## Sentiment Distribution Across Subreddits

In [89]:

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# ── 4. Sentiment Analysis ────────────────────────────────────────────────────
print("\n[4] Running VADER Sentiment Analysis …")

# 1. Download the required lexicon (only needs to run once)
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')



[4] Running VADER Sentiment Analysis …


In [97]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── 4. Sentiment Analysis Distribution ────────────────────────────────────────
print("\n[4] Plotting sentiment distribution from 'self_text' VADER scores …")

# FIX: Explicitly build the sentiment_label column if it doesn't exist yet
if "sentiment_label" not in sample_df.columns:
    print("   Building 'sentiment_label' column from 'vader_compound' …")
    
    def categorical_sentiment(score):
        if score >= 0.05:
            return "Positive"
        elif score <= -0.05:
            return "Negative"
        else:
            return "Neutral"
            
    # Map the existing compound scores to labels
    sample_df["sentiment_label"] = sample_df["vader_compound"].apply(categorical_sentiment)

# 1. Filter dataset down to your target communities
sub_df = sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]

if sub_df.empty:
    print("Warning: No matching subreddits found in FOCUS_SUBREDDITS. Check your list filter.")
else:
    # 2. Group by subreddit and calculate percentage distributions
    sentiment_counts = (
        sub_df.groupby("subreddit")["sentiment_label"]
        .value_counts(normalize=True)
        .unstack()
        .fillna(0) * 100
    )

    # 3. Ensure columns align cleanly (Negative -> Neutral -> Positive)
    sentiment_counts = sentiment_counts.reindex(columns=["Negative", "Neutral", "Positive"])

    # 4. Generate the 100% stacked horizontal bar chart
    fig, ax = plt.subplots(figsize=(13, 7))

    sentiment_counts.plot(
        kind="barh", 
        stacked=True, 
        color=["#e74c3c", "#95a5a6", "#2ecc71"],  # Soft Red, Gray, Soft Green
        ax=ax,
        width=0.6
    )

    # Adjust styling & labels
    ax.set_title("Emotional Tone Distribution Across Focus Subreddits", fontsize=13, fontweight="bold")
    ax.set_xlabel("Percentage (%)", fontsize=11)
    ax.set_ylabel("Subreddit", fontsize=11)
    ax.legend(title="Sentiment", bbox_to_anchor=(1.05, 1), loc='upper left')

    # 5. Dynamic text rendering inside the bars
    for p in ax.patches:
        width = p.get_width()
        if width > 5:  # Only add readable text labels to blocks wider than 5%
            x = p.get_x() + width / 2
            y = p.get_y() + p.get_height() / 2
            ax.text(x, y, f"{width:.1f}%", ha='center', va='center', color='white', fontweight='bold', fontsize=9)

    plt.tight_layout()
    plt.show()
    save(fig, "05_subreddit_sentiment_distribution.png")


[4] Plotting sentiment distribution from 'self_text' VADER scores …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\05_subreddit_sentiment_distribution.png


### Advanced Context-Sentiment Cross
Since we have both self_text and vader_compound columns available, you can easily filter your context phrases by tone. For instance, if you want to find what specifically drives negativity within r/politics or r/conspiracy

In [98]:
# Quick analysis filter example:
negative_politics = sub_df[(sub_df["subreddit"] == "politics") & (sub_df["sentiment_label"] == "Negative")]
# You can now pass negative_politics["tokens"] into your context discovery code block!

In [99]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from nltk.util import ngrams

# ── 5. Sentimental Context Discovery ──────────────────────────────────────────
print("\n[5] Extracting context filtered by emotional tone …")

# Re-using your clean context extractor from earlier
def get_valid_ngrams(token_list, n=2):
    generated_ngrams = ngrams(token_list, n)
    valid_phrases = []
    for gram in generated_ngrams:
        phrase = " ".join(gram)
        if any(len(word.strip()) <= 1 for word in gram if word.isalnum()): 
            continue
        valid_phrases.append(phrase)
    return valid_phrases

# 1. Filter data by target subreddit AND emotional tone
# Let's target 'conspiracy' as an example from your dataset
TARGET_SUB = "conspiracy"
sub_focused_df = sample_df[sample_df["subreddit"] == TARGET_SUB]

# 2. Separate tokens based on sentiment labels
neg_tokens_series = sub_focused_df[sub_focused_df["sentiment_label"] == "Negative"]["tokens"]
pos_tokens_series = sub_focused_df[sub_focused_df["sentiment_label"] == "Positive"]["tokens"]

# 3. Extract Bigrams (N=2) for both subsets
neg_phrases = [p for ts in neg_tokens_series for p in get_valid_ngrams(ts, n=2)]
pos_phrases = [p for ts in pos_tokens_series for p in get_valid_ngrams(ts, n=2)]

# 4. Aggregate top 15 phrases for each side
top_neg_phrases = Counter(neg_phrases).most_common(15)
top_pos_phrases = Counter(pos_phrases).most_common(15)

# ── 6. Plotting Side-by-Side Context ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Plot Negative Context
if top_neg_phrases:
    phrases_n, counts_n = zip(*top_neg_phrases)
    ax1.barh(phrases_n[::-1], counts_n[::-1], color=sns.color_palette("Reds_r", len(top_neg_phrases)))
    ax1.set_title(f"Negative Context Phrases in r/{TARGET_SUB}", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Phrase Count")

# Plot Positive Context
if top_pos_phrases:
    phrases_p, counts_p = zip(*top_pos_phrases)
    ax2.barh(phrases_p[::-1], counts_p[::-1], color=sns.color_palette("YlGn_r", len(top_pos_phrases)))
    ax2.set_title(f"Positive Context Phrases in r/{TARGET_SUB}", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Phrase Count")

plt.suptitle(f"Thematic Contrast Analysis for r/{TARGET_SUB}", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
save(fig, f"06_G_sentimental_context_{TARGET_SUB}.png")


[5] Extracting context filtered by emotional tone …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\06_G_sentimental_context_conspiracy.png


In [52]:

# ── 4. Controversiality ────────────────────────────────────────────────────────
print("\n[4] Controversiality analysis …")

# 4a. Rate by subreddit (full df)
cont_rate = (
    df[df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby("subreddit")["controversial"]
    .agg(["mean","sum","count"]).reset_index()
    .rename(columns={"mean":"controversy_rate","sum":"n_controversial","count":"n_total"})
    .sort_values("controversy_rate", ascending=False)
)
cont_rate.to_csv(os.path.join(OUT_DIR, "05_controversiality_by_subreddit.csv"), index=False)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(cont_rate["subreddit"][::-1], cont_rate["controversy_rate"][::-1] * 100,
        color=sns.color_palette("Reds_r", len(cont_rate)))
ax.set_xlabel("Controversy Rate (%)", fontsize=11)
ax.set_title("Controversiality Rate by Subreddit", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
save(fig, "05_controversiality_by_subreddit.png")

# 4b. Sentiment vs controversiality (use sample_df)
cont_sent = (
    sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby(["subreddit","controversial"])["vader_compound"]
    .mean().unstack().reset_index()
)
cont_sent.columns = ["subreddit","not_controversial","controversial"]
cont_sent = cont_sent.dropna().sort_values("not_controversial")

x = np.arange(len(cont_sent)); w = 0.35
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - w/2, cont_sent["not_controversial"], w, label="Not controversial", color="#4C72B0")
ax.bar(x + w/2, cont_sent["controversial"],     w, label="Controversial",     color="#d62728")
ax.set_xticks(x)
ax.set_xticklabels(cont_sent["subreddit"], rotation=40, ha="right", fontsize=8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Mean VADER Compound Score", fontsize=11)
ax.set_title("Sentiment: Controversial vs. Non-Controversial Comments", fontsize=12, fontweight="bold")
ax.legend(); plt.tight_layout()
save(fig, "05_sentiment_vs_controversiality.png")

# 4c. Controversiality over time (full df)
cont_monthly = (
    df.dropna(subset=["comment_month"])
    .groupby("comment_month")["controversial"]
    .agg(["mean","sum","count"]).reset_index()
)
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(cont_monthly["comment_month"], cont_monthly["sum"], width=20,
        color="#d62728", alpha=0.5, label="# Controversial comments")
ax1.set_ylabel("Count", color="#d62728", fontsize=11)
ax1.tick_params(axis="y", labelcolor="#d62728")
ax2 = ax1.twinx()
ax2.plot(cont_monthly["comment_month"], cont_monthly["mean"] * 100,
         color="#333333", marker="o", linewidth=1.5, label="Rate (%)")
ax2.set_ylabel("Controversy Rate (%)", fontsize=11)
ax1.set_title("Controversiality Over Time", fontsize=13, fontweight="bold")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.xticks(rotation=30); plt.tight_layout()
save(fig, "05_controversiality_trend.png")

# 4d. Top keywords: controversial vs non-controversial
for label_val, grp_df, tag in [
    (1, cont1, "controversial"),
    (0, cont0, "non_controversial"),
]:
    toks = [t for ts in grp_df["tokens"] for t in ts]
    top20 = Counter(toks).most_common(20)
    words_l, counts_l = zip(*top20)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(words_l[::-1], counts_l[::-1], color="#d62728" if label_val==1 else "#4C72B0")
    ax.set_title(f"Top Keywords — {'Controversial' if label_val==1 else 'Non-Controversial'} Comments",
                 fontsize=12, fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
    plt.tight_layout()
    save(fig, f"06_keywords_{tag}.png")

# ── 5. Summary ─────────────────────────────────────────────────────────────────
print("\n[5] Writing summary CSV …")
summary = (
    df[df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby("subreddit")
    .agg(
        n_comments       = ("comment_id",       "count"),
        mean_score       = ("score",            "mean"),
        median_score     = ("score",            "median"),
        n_controversial  = ("controversial",    "sum"),
        controversy_rate = ("controversial",    "mean"),
        mean_word_count  = ("comment_word_count","mean"),
    )
    .reset_index().round(4)
)
summary.to_csv(os.path.join(OUT_DIR, "00_subreddit_summary.csv"), index=False)
print("  Saved → figures/00_subreddit_summary.csv")
print("\n✓ All done.")



[4] Controversiality analysis …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\05_controversiality_by_subreddit.png
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\05_sentiment_vs_controversiality.png
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\05_controversiality_trend.png
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\06_keywords_controversial.png
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\06_keywords_non_controversial.png

[5] Writing summary CSV …
  Saved → figures/00_subreddit_summary.csv

✓ All done.


In [100]:

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# ── 5. Topic Modeling (LDA) ──────────────────────────────────────────────────
print("\n[5] Commencing Latent Dirichlet Allocation (Topic Modeling) …")

# 1. Configuration Parameters
N_TOPICS = 5          # Number of hidden clusters/themes you want to uncover
N_TOP_WORDS = 10       # Number of descriptive keywords to display per topic

# 2. Reconstruct token lists back into flat text documents for Scikit-learn
print("   Reconstituting text collections …")
corpus = sample_df["tokens"].apply(lambda tokens: " ".join([str(t) for t in tokens]))

# 3. Vectorize text down to a Document-Term Matrix
# max_df=0.95 means ignore terms that appear in more than 95% of documents (too generic)
# min_df=2 means ignore terms that appear in fewer than 2 documents (typos/outliers)
vectorizer = CountVectorizer(max_df=0.95, min_df=2)
dtm = vectorizer.fit_transform(corpus)

# 4. Initialize and fit the LDA Model
print(f"   Fitting LDA model with {N_TOPICS} topics (80K max sample context) …")
lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS, 
    max_iter=10,            # Set to 10-15 iterations for balanced performance/accuracy
    learning_method='online', 
    random_state=42,
    n_jobs=-1               # Use all available CPU cores
)
lda_model.fit(dtm)

# 5. Extract and Plot top terms driving each theme structural column
print("   Plotting isolated structural cluster themes …")
feature_names = vectorizer.get_feature_names_out()

# Setup grid system for multi-topic parsing
fig, axes = plt.subplots(1, N_TOPICS, figsize=(20, 6), sharex=False)
axes = axes.flatten()

for topic_idx, topic in enumerate(lda_model.components_):
    # Get indices of the highest weight terms for this topic
    top_word_indices = topic.argsort()[:-N_TOP_WORDS - 1:-1]
    top_words = [feature_names[i] for i in top_word_indices]
    top_weights = [topic[i] for i in top_word_indices]
    
    # Plot horizontal weights
    ax = axes[topic_idx]
    ax.barh(top_words[::-1], top_weights[::-1], color=sns.color_palette("muted")[topic_idx % 6])
    ax.set_title(f"Topic Theme #{topic_idx + 1}", fontsize=12, fontweight="bold")
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.grid(axis='x', linestyle='--', alpha=0.5)

plt.suptitle(f"Top Unsupervised Topic Modeling Clusters (LDA Model)", fontsize=15, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()
save(fig, "07_unsupervised_lda_topics.png")


[5] Commencing Latent Dirichlet Allocation (Topic Modeling) …
   Reconstituting text collections …
   Fitting LDA model with 5 topics (80K max sample context) …
   Plotting isolated structural cluster themes …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\07_unsupervised_lda_topics.png
